# Importing Modules and Defining Necessary Functions

In [8]:
import numpy as np
from datetime import datetime
import os
import requests
import torch
from torch import nn
from torch.nn import functional as F
import numpy as np
from dataclasses import dataclass, asdict
import TransformerMain


def loadModel(pathReq):
    # model.load_state_dict(torch.load(pathReq, weights_only=True))        
    # model.eval()
    ckpt = torch.load(pathReq)    
    cfg = Configs(**ckpt["config"])
    model = TransformerMain.Transformer(cfg)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    
    return model
    
def myDataLoader(dataSet, batchSize, blockSize):
  batch = []
  toPredict = []
  i = 0
  block = 0
  toShuffle = np.arange(0,batchSize)
  np.random.shuffle(toShuffle)
  
  for b in range(0, len(dataSet)//blockSize):
      
    batch.append(dataSet[block : (block + blockSize)])
    toPredict.append(dataSet[block + 1 : (block + 1 + blockSize)]) #adding next letter for the one to predict
    block += blockSize
    if (b+1) % batchSize == 0:
      # yield batch, toPredict      
      yield np.array(batch)[toShuffle], np.array(toPredict)[toShuffle]
      batch = []
      toPredict = []


def getRawDataWords():
  data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
  data = requests.get(data_url).text.split() #do it per word

  print(f"length of raw dataset(USING WORDS NOT CHARS): {len(data):,}")

  # get all the unique characters that occur in this text
  possibleTokens = sorted(list(set(data)))

  return data, possibleTokens, len(possibleTokens)
    
wordToTensor = lambda sInput, tokenMapping: [tokenMapping[letter] for letter in sInput] # to get numerical tensor to feed into nn.Embedding function (each letter has it's index)
tensorToWord = lambda sIndexes, tokenMapping: [list(tokenMapping.keys())[i.item()] for i in sIndexes]
data, gTokens, gVocabSize = getRawDataWords()
vocab = gTokens 
tokenMapping = {k:v for v, k in enumerate(vocab)} #make dict where value is index of token and key is token itself
numericRepresentation = wordToTensor(data,tokenMapping) #changing character to index-based integer
        

length of raw dataset(USING WORDS NOT CHARS): 202,651


In [9]:
def generateGreedyWithKVForWords(model, prompt, tokenMapping, config, max_gen = 512):
    print(prompt)
    x =  torch.tensor(wordToTensor(prompt,tokenMapping))[None].to(device=config.device)
    model = model.eval()
    
    with torch.no_grad():        
        logits, past = model(x)    
        out = torch.cat([x, logits[:,[-1],:].argmax(dim = -1)], dim = 1)
        
        for i in range(max_gen-x.shape[1]):              
            logits, _ = model(out[:,[-1]], i+x.shape[1]) #only need to calculate one token's importance score             
            next_token_index = torch.argmax(logits, dim = -1)[:,[-1]]   # greedy sampling             
            out = torch.cat([out, next_token_index], dim = -1)
            

    return ' '.join(tensorToWord(out.squeeze(0), tokenMapping))

# Initializing Configs

In [10]:
att_dict = {'mha' : 'multi_head_attention', 
            'window': 'sliding_window_attention', 
            'mqa': 'multi_query_attention',
            'gqa': 'grouped_query_attention',
            'flash': 'flash_attention',
           'paged': 'paged_attention'}

ff_dict = {'relu' : 'mlp_with_relu', 'gelu': 'mlp_with_gelu', 'swiglu': 'mlp_with_swiglu'}
lr_decay = {'cosine': 'cosine', 'linear':'linear'}
@dataclass
class Configs:
    device: str = 'cpu' if not torch.cuda.is_available() else 'cuda'
    vocab_size: int = 65
    seq_length: int = 512    
    max_seq_length: int = 1024
    batch_size: int = 32
    d_model: int = 64
    num_heads: int = 8
    num_blocks: int = 4
    num_groups: int = 4 
    attention_type: str = att_dict['gqa']
    ff: str = ff_dict['relu']
    lr_warmup: bool = True
    lr_decay: str = lr_decay['cosine']
    gradient_clipping: bool = True
    inference: bool = False
    window_size: int = 128
    epochs: int = 100
    pos_embed: str = 'rope' #could be rope or sinusoidal
    use_experts: bool = True    
    num_experts: int = 4
    top_k_experts: int = 2

config = Configs()

# Loading a small model and the larger one

In [22]:

prompt = ['First']

fileName = os.path.join(os.getcwd(), 'SavedModels/model_MoE_words_cheap.pth')
smallModel = loadModel(fileName).to(device=config.device)
smallModel.config.inference = True 
out_with_kv = generateGreedyWithKVForWords(smallModel, prompt, tokenMapping, smallModel.config, smallModel.config.seq_length)

fileName = os.path.join(os.getcwd(), 'SavedModels/model_MoE_words_target.pth')
target = loadModel(fileName)


['First']


# Running larger model to get outputs

In [23]:
prompt = out_with_kv.split()
x =  torch.tensor(wordToTensor(prompt,tokenMapping))[None].to(device=config.device)
target = target.cuda().eval()
logits, _ = target(x[:, :])
next_indices = torch.argmax(logits, dim = -1)
out = torch.cat([x, next_indices], dim = -1)

# Demonstrating Speculative Decoding Loop

In [24]:
unmasked = ''
futureTokenFromLargeModel = tensorToWord(x.squeeze(0), tokenMapping)[0] # initializing to first word of small model as it doesn't count
i = 0
for wordSmallModel, wordLargeModel in zip(tensorToWord(x.squeeze(0), tokenMapping), tensorToWord(next_indices.squeeze(0), tokenMapping)):
    unmasked +=  wordSmallModel + ' '
    print('=' * 100)    
    print(f'Small model next token:"{wordSmallModel}" | Larger model next token:"{futureTokenFromLargeModel}"')   
    print('Same prediction as smaller model?', wordSmallModel == futureTokenFromLargeModel)
    print('=' * 100)    
    # print(unmasked, '| Larger model prediction:', wordLargeModel) #p(next_token|previous_tokens)
    print(f' \n >>> "{unmasked}"\n')
    
    if wordSmallModel != futureTokenFromLargeModel:
        print(f'\n\nLarger model differs, so fall back to larger model at token {i}\n\n')
        break
    
    futureTokenFromLargeModel = wordLargeModel
    
    i += 1

Small model next token:"First" | Larger model next token:"First"
Same prediction as smaller model? True
 
 >>> "First "

Small model next token:"Citizen:" | Larger model next token:"Citizen:"
Same prediction as smaller model? True
 
 >>> "First Citizen: "

Small model next token:"Before" | Larger model next token:"Before"
Same prediction as smaller model? True
 
 >>> "First Citizen: Before "

Small model next token:"we" | Larger model next token:"we"
Same prediction as smaller model? True
 
 >>> "First Citizen: Before we "

Small model next token:"proceed" | Larger model next token:"proceed"
Same prediction as smaller model? True
 
 >>> "First Citizen: Before we proceed "

Small model next token:"any" | Larger model next token:"any"
Same prediction as smaller model? True
 
 >>> "First Citizen: Before we proceed any "

Small model next token:"further," | Larger model next token:"further,"
Same prediction as smaller model? True
 
 >>> "First Citizen: Before we proceed any further, "

Sma